# Chapter 24 Companion Notebook: Transformers and Large Language Models

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch24_Transformers_and_Large_Language_Models.ipynb)

This notebook accompanies Chapter 24 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




This notebook is designed as a classroom appendix. It uses synthetic business text data so that every step can run safely in Google Colab without external files, paid APIs, or private customer information. The goal is not to train a production large language model. The goal is to show how analysts reason about tokens, context windows, self-attention, encoder-style text classification, decoder-style structured outputs, retrieval-supported workflows, reliability tests, and governance controls.


## Why this matters (business framing)

Transformer models and large language models are useful in business analytics because they convert messy language into decision-support artifacts: labels, extracted fields, summaries, similarity matches, evidence packages, and analyst-facing answers. The managerial risk is that these outputs can sound confident even when the system omitted evidence, used the wrong context, ignored a required schema, or answered outside its approved scope.

This notebook treats transformer-based NLP as a workflow design problem. We will start with tokenization and context length, build a small self-attention example, train a tiny encoder-style Transformer for issue classification, create structured output patterns that mimic decoder-style use, build a lightweight retrieval-supported answer package, evaluate failure modes, and document governance controls. The code intentionally avoids paid APIs so that the mechanics remain visible and reproducible.


## Agenda

1. Setup and reproducibility
2. Synthetic business text corpus and knowledge base
3. Tokens, subwords, context length, truncation, and chunking
4. Self-attention, causal masking, and contextual meaning
5. Encoder-style classification with a tiny Transformer
6. Decoder-style structured outputs without calling a paid API
7. Retrieval-supported question answering package
8. Reliability tests for business failure modes
9. Token budget, batch workflow design, and ROI simulation
10. Governance checklist, system card, and exercises


## Connection map

Chapters 20 through 23 introduced the foundations needed for reliable text analytics: corpus quality, embeddings and similarity, topic discovery, sentiment measurement, and supervised classification. Chapter 24 extends those ideas to transformer-based NLP and foundation models. A more capable model does not remove the earlier design requirements. Analysts still need to define the unit of analysis, choose the allowed output schema, test by business-relevant slices, prevent leakage, and monitor failures over time.

Chapter 25 will go deeper into retrieval-augmented text mining. This notebook introduces the basic retrieval-supported workflow because retrieval is one of the most practical ways to ground generative systems in approved business evidence.


In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================

import os
# Keep classroom notebooks lightweight and prevent BLAS/OpenMP oversubscription in small CPU environments.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import importlib.util
import subprocess
import sys
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "torch": "torch",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import math
import random
import re
import time
import warnings
from collections import Counter, defaultdict
from datetime import timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("ch24_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")


## Utility functions

These helper functions keep the main sections focused on business analytics rather than boilerplate. They support text normalization, approximate tokenization, model evaluation, compact displays, attention visualization, retrieval, and output validation.


In [ ]:
# ============================================================
# Utility functions for transformer-style text analytics
# ============================================================

PRODUCT_NAMES = ["NovaPhone", "FitBand", "CloudHome", "AtlasBook", "ShopEasy"]
ISSUE_TYPES = ["billing", "power", "delivery", "login", "product_quality", "praise"]
ALLOWED_ISSUES = set(ISSUE_TYPES).union({"out_of_scope"})


def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", " URL ", text)
    text = re.sub(r"[^a-z0-9#@_\-\s'!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def word_tokenize(text):
    return re.findall(r"[a-z0-9_#@\-']+|[!?]", normalize_text(text))


def approximate_subword_tokenize(text):
    """A tiny teaching tokenizer. It is not a production tokenizer.

    The function keeps hashtags and product codes visible, then breaks long or uncommon-looking
    tokens into pseudo-subword pieces to illustrate why token count differs from word count.
    """
    raw = re.findall(r"#[A-Za-z0-9_]+|@[A-Za-z0-9_]+|[A-Za-z]+\d+[A-Za-z0-9\-]*|[A-Za-z]+|\d+|[^\w\s]", str(text))
    pieces = []
    for token in raw:
        if re.match(r"#[A-Za-z0-9_]+", token):
            pieces.append("#")
            body = token[1:]
            if len(body) > 8:
                pieces.extend([body[:6].lower(), "##" + body[6:].lower()])
            else:
                pieces.append(body.lower())
        elif re.match(r"[A-Za-z]+\d+", token) or "-" in token:
            parts = re.findall(r"[A-Za-z]+|\d+|\-", token)
            for j, part in enumerate(parts):
                pieces.append(part.lower() if j == 0 else "##" + part.lower())
        elif token.isalpha() and len(token) > 10:
            pieces.append(token[:6].lower())
            pieces.append("##" + token[6:].lower())
        else:
            pieces.append(token.lower())
    return pieces


def token_count(text):
    return len(approximate_subword_tokenize(text))


def chunk_tokens(tokens, max_tokens=80, overlap=15):
    if max_tokens <= 0:
        raise ValueError("max_tokens must be positive")
    if overlap >= max_tokens:
        raise ValueError("overlap must be smaller than max_tokens")
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunks.append(tokens[start:end])
        if end == len(tokens):
            break
        start = end - overlap
    return chunks


def compact_corpus_profile(df, rows=8):
    print(f"Rows: {len(df):,}")
    print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"Issue labels: {', '.join(sorted(df['issue_type'].unique()))}")
    print(f"Mean approximate token count: {df['token_count'].mean():.1f}")
    display(df.sample(min(rows, len(df)), random_state=SEED)[[
        "record_id", "date", "channel", "product", "issue_type", "sentiment", "risk_level", "text"
    ]])


def plot_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, values_format="d")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()


def scaled_dot_product_attention(Q, K, V, mask=None):
    scores = Q @ K.T / math.sqrt(Q.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    scores = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores)
    weights = weights / weights.sum(axis=-1, keepdims=True)
    output = weights @ V
    return output, weights


def plot_attention(tokens, weights, title="Attention weights"):
    fig, ax = plt.subplots(figsize=(max(6, len(tokens) * 0.55), max(5, len(tokens) * 0.45)))
    im = ax.imshow(weights)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_xlabel("Attended token")
    ax.set_ylabel("Query token")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def validate_structured_output(obj, allowed_issues=ALLOWED_ISSUES):
    required = {
        "issue_type": str,
        "urgency": str,
        "product": str,
        "evidence_span": str,
        "confidence": float,
        "needs_human_review": bool,
    }
    problems = []
    for key, expected_type in required.items():
        if key not in obj:
            problems.append(f"missing field: {key}")
        elif not isinstance(obj[key], expected_type):
            problems.append(f"field {key} has type {type(obj[key]).__name__}, expected {expected_type.__name__}")
    if obj.get("issue_type") not in allowed_issues:
        problems.append(f"issue_type not allowed: {obj.get('issue_type')}")
    if obj.get("urgency") not in {"low", "medium", "high"}:
        problems.append(f"urgency not allowed: {obj.get('urgency')}")
    conf = obj.get("confidence", 0.0)
    if not isinstance(conf, float) or not (0.0 <= conf <= 1.0):
        problems.append("confidence must be a float between 0 and 1")
    return problems


def first_matching_sentence(text, keywords):
    sentences = re.split(r"(?<=[.!?])\s+", str(text).strip())
    lowered_keywords = [k.lower() for k in keywords]
    for sent in sentences:
        s = sent.lower()
        if any(k in s for k in lowered_keywords):
            return sent.strip()
    return sentences[0].strip() if sentences else ""


## 2. Synthetic business text corpus and knowledge base

The dataset below mimics a combined repository of reviews, support tickets, chat messages, and survey comments. Each row is a text observation with metadata, a known issue label, sentiment, a risk indicator, and a business timestamp. The knowledge base is a small set of approved policy passages that a retrieval-supported workflow can use later in the notebook.


In [ ]:
# ============================================================
# 2.1 Generate a synthetic corpus of business texts
# ============================================================

ISSUE_TEMPLATES = {
    "billing": [
        "The charge on my bill is wrong for {product}. I need a refund for order {code}.",
        "I was billed twice after upgrading {product}. Please fix the invoice before renewal.",
        "The subscription fee for {product} looks higher than promised and the receipt is confusing.",
        "Not a product defect, but the charge on the card is incorrect for {product}.",
    ],
    "power": [
        "The battery charge on my {product} drops fast even after a full night on the charger.",
        "My {product} will not hold a charge and the power indicator keeps blinking.",
        "I like the design, but the charger for {product} overheats after twenty minutes.",
        "The battery is not bad at first, but by afternoon the charge is almost gone.",
    ],
    "delivery": [
        "The package for {product} arrived late and the tracking page still says pending.",
        "My {product} order {code} was marked delivered, but it is not at my door.",
        "Shipping was delayed twice and customer support could not explain the new delivery date.",
        "The box arrived damaged even though the delivery notification said everything was fine.",
    ],
    "login": [
        "I cannot log in to manage my {product} account after the password reset.",
        "The app keeps signing me out and two-factor verification fails on {product}.",
        "My account is locked even though I entered the correct code for order {code}.",
        "The login screen freezes when I try to update my subscription settings.",
    ],
    "product_quality": [
        "The screen on my {product} cracked during normal use and the sensor is unreliable.",
        "The button on {product} stopped working after a week and the build feels cheap.",
        "I received {product} with a scratched case, loose hinge, and inconsistent audio.",
        "The product is usable, but the quality is not what the campaign promised.",
    ],
    "praise": [
        "The {product} setup was easy, delivery was smooth, and the app is helpful.",
        "I love my {product}. The battery lasts long and the support team answered quickly.",
        "Great experience with {product}; the price was clear and the checkout flow was simple.",
        "Not bad at all. {product} works as expected and the instructions are clear.",
    ],
}

CHANNELS = ["review", "support_ticket", "chat", "survey"]
SEGMENTS = ["new_customer", "loyal_customer", "enterprise", "price_sensitive"]
PRODUCT_WEIGHTS = [0.28, 0.20, 0.18, 0.16, 0.18]
ISSUE_WEIGHTS = [0.20, 0.17, 0.16, 0.16, 0.17, 0.14]
URGENCY_PHRASES = ["urgent", "today", "escalate", "manager", "cancel", "safety", "deadline"]


def make_order_code(rng):
    prefix = rng.choice(["NX", "FB", "CH", "AB", "SE"])
    return f"{prefix}-{rng.integers(1000, 9999)}"


def make_synthetic_corpus(n=900, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    start_date = pd.Timestamp("2026-01-01")
    for i in range(n):
        issue = rng.choice(ISSUE_TYPES, p=np.array(ISSUE_WEIGHTS) / np.sum(ISSUE_WEIGHTS))
        product = rng.choice(PRODUCT_NAMES, p=np.array(PRODUCT_WEIGHTS) / np.sum(PRODUCT_WEIGHTS))
        channel = rng.choice(CHANNELS, p=[0.30, 0.34, 0.20, 0.16])
        segment = rng.choice(SEGMENTS, p=[0.32, 0.30, 0.18, 0.20])
        template = rng.choice(ISSUE_TEMPLATES[issue])
        code = make_order_code(rng)
        text = template.format(product=product, code=code)

        # Add channel-specific and temporal noise that resembles real business text.
        if channel == "chat" and rng.random() < 0.45:
            text = "Hi, " + text + " Can someone help?"
        if channel == "support_ticket" and rng.random() < 0.40:
            text = text + f" Ticket ref @{segment}_{rng.integers(10, 99)}."
        if rng.random() < 0.22:
            text = text + " #CustomerCare"
        if rng.random() < 0.12:
            text = text + " 😊" if issue == "praise" else text + " 😟"
        if issue != "praise" and rng.random() < 0.18:
            text = text + " This is urgent and I may cancel if it is not resolved."
        if issue == "praise" and rng.random() < 0.20:
            text = text + " I would recommend it to a friend."

        date = start_date + pd.Timedelta(days=int(rng.integers(0, 180)))
        sentiment = "positive" if issue == "praise" else rng.choice(["negative", "mixed", "neutral"], p=[0.62, 0.30, 0.08])
        risk_level = "high" if any(w in text.lower() for w in ["urgent", "cancel", "safety", "manager"]) else ("low" if issue == "praise" else "medium")
        rows.append({
            "record_id": f"TXT-{i+1:04d}",
            "date": date,
            "channel": channel,
            "segment": segment,
            "product": product,
            "issue_type": issue,
            "sentiment": sentiment,
            "risk_level": risk_level,
            "text": text,
        })
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    df["text_clean"] = df["text"].map(normalize_text)
    df["token_count"] = df["text"].map(token_count)
    return df


df = make_synthetic_corpus(n=900, seed=SEED)
compact_corpus_profile(df)


In [ ]:
# ============================================================
# 2.2 Create a small approved knowledge base for retrieval-supported workflows
# ============================================================

knowledge_base = pd.DataFrame([
    {
        "doc_id": "KB-REFUND-001",
        "topic": "billing",
        "title": "Refund and invoice correction policy",
        "passage": "When a customer reports a duplicate charge, incorrect invoice, or subscription fee mismatch, verify the order code, compare the promised price with the billing ledger, and offer a refund or invoice correction if the mismatch is confirmed.",
    },
    {
        "doc_id": "KB-POWER-002",
        "topic": "power",
        "title": "Battery and charger troubleshooting",
        "passage": "For battery drain, charger overheating, or failure to hold a charge, collect the device model, charging duration, power indicator behavior, and whether the customer used an approved charger. Escalate safety complaints immediately.",
    },
    {
        "doc_id": "KB-SHIP-003",
        "topic": "delivery",
        "title": "Delayed or missing delivery process",
        "passage": "For late, missing, or damaged deliveries, check tracking status, carrier scan history, delivery address, and package photo evidence. Create a replacement request when the carrier confirms loss or damage.",
    },
    {
        "doc_id": "KB-LOGIN-004",
        "topic": "login",
        "title": "Account login and verification recovery",
        "passage": "For login failures, password reset loops, account locks, or two-factor verification errors, verify the account email, reset token age, device type, and recent security changes before escalating to identity support.",
    },
    {
        "doc_id": "KB-QUALITY-005",
        "topic": "product_quality",
        "title": "Product defect and warranty intake",
        "passage": "For cracked screens, broken buttons, loose hinges, sensor failures, or damaged items on arrival, collect photos, purchase date, serial number, and usage context. Warranty eligibility depends on defect type and normal-use evidence.",
    },
    {
        "doc_id": "KB-SCOPE-006",
        "topic": "governance",
        "title": "Approved use boundaries",
        "passage": "The text analytics assistant may classify customer feedback, extract evidence spans, retrieve approved policy passages, and prepare triage summaries. It must not provide legal, medical, financial, or employment advice.",
    },
])

display(knowledge_base)


## 3. Tokens, subwords, context length, truncation, and chunking

Transformer systems operate on tokens, not visible words. A brand name, hashtag, emoji, product code, or uncommon phrase can use more tokens than a student might expect. Token count affects context limits, latency, and cost. Long inputs also create measurement decisions: should the analyst truncate, summarize, chunk, or retrieve only the relevant passages?


In [ ]:
# ============================================================
# 3.1 Compare visible words with approximate subword tokens
# ============================================================

examples = [
    "NovaPhone #BatteryLife 😊 order NX-90210 has a weird overcharging issue.",
    "The charge on my bill is wrong, but the battery charge is fine.",
    "CloudHomeSuperRouterPro disconnects after two-factor verification.",
    "Not bad at all, but the ShopEasy subscription renewal page is confusing.",
]

profile_rows = []
for text in examples:
    words = text.split()
    toks = approximate_subword_tokenize(text)
    profile_rows.append({
        "text": text,
        "visible_word_count": len(words),
        "approx_token_count": len(toks),
        "tokens": toks,
    })

token_profile = pd.DataFrame(profile_rows)
display(token_profile)


In [ ]:
# ============================================================
# 3.2 Context length and chunking demonstration
# ============================================================

long_ticket = (
    "Customer wrote a long support ticket. "
    + "The setup guide was clear and the first week was smooth. " * 12
    + "However, the final section says the invoice charge is wrong and the subscription renewal fee doubled. "
    + "The customer wants a refund today."
)

long_tokens = approximate_subword_tokenize(long_ticket)
for budget in [40, 80, 120]:
    visible = " ".join(long_tokens[:budget])
    contains_late_billing = "invoice" in visible or "refund" in visible or "subscription" in visible
    print(f"Budget {budget:>3} tokens | contains late billing evidence? {contains_late_billing}")

chunks = chunk_tokens(long_tokens, max_tokens=60, overlap=12)
chunk_table = pd.DataFrame({
    "chunk_id": [f"chunk_{i+1}" for i in range(len(chunks))],
    "n_tokens": [len(c) for c in chunks],
    "contains_billing_evidence": [any(t in c for t in ["invoice", "refund", "subscription", "charge"]) for c in chunks],
    "preview": [" ".join(c[:18]) + (" ..." if len(c) > 18 else "") for c in chunks],
})
display(chunk_table)


In [ ]:
# ============================================================
# 3.3 Corpus-level token budget profile
# ============================================================

token_summary = df.groupby("channel").agg(
    n=("record_id", "count"),
    mean_tokens=("token_count", "mean"),
    p90_tokens=("token_count", lambda x: np.percentile(x, 90)),
    max_tokens=("token_count", "max"),
).round(1).reset_index()
display(token_summary)

plt.figure(figsize=(8, 4))
plt.hist(df["token_count"], bins=24)
plt.title("Approximate token count distribution")
plt.xlabel("Approximate token count")
plt.ylabel("Number of texts")
plt.tight_layout()
plt.show()


## 4. Self-attention, causal masking, and contextual meaning

Self-attention lets each token compare itself with other tokens in the same sequence. Encoder-style models usually allow tokens to attend bidirectionally across the input. Decoder-style generation uses causal masking so that a token can only attend to earlier positions while predicting the next token. The small examples below are educational illustrations of the mechanics, not production model internals.


In [ ]:
# ============================================================
# 4.1 Scaled dot-product attention on a business sentence
# ============================================================

sentence = "the charge on my bill is wrong but the battery charge is fine"
tokens = sentence.split()

rng = np.random.default_rng(SEED)
d_model = 8
base_vectors = {tok: rng.normal(scale=0.20, size=d_model) for tok in sorted(set(tokens))}

# Add simple semantic directions so that billing and power cues are visible in the heatmap.
billing_direction = np.array([1, 0, 0, 0, 0, 0, 0, 0], dtype=float)
power_direction = np.array([0, 1, 0, 0, 0, 0, 0, 0], dtype=float)
contrast_direction = np.array([0, 0, 1, 0, 0, 0, 0, 0], dtype=float)
for tok in ["bill", "wrong"]:
    base_vectors[tok] = base_vectors[tok] + billing_direction
for tok in ["battery", "fine"]:
    base_vectors[tok] = base_vectors[tok] + power_direction
for tok in ["but"]:
    base_vectors[tok] = base_vectors[tok] + contrast_direction
base_vectors["charge"] = base_vectors["charge"] + 0.45 * billing_direction + 0.45 * power_direction

X = np.vstack([base_vectors[tok] for tok in tokens])
Q, K, V = X, X, X
context_vectors, attention_weights = scaled_dot_product_attention(Q, K, V)
plot_attention(tokens, attention_weights, title="Toy self-attention over an ambiguous business sentence")

charge_positions = [i for i, tok in enumerate(tokens) if tok == "charge"]
print("Positions of the repeated token 'charge':", charge_positions)
print("Attention paid by each 'charge' token to billing and power cues:")
for pos in charge_positions:
    row = attention_weights[pos]
    print({tok: round(float(row[i]), 3) for i, tok in enumerate(tokens) if tok in {"bill", "wrong", "battery", "fine", "charge"}})


In [ ]:
# ============================================================
# 4.2 Encoder-style full attention versus decoder-style causal masking
# ============================================================

n = len(tokens)
full_mask = np.ones((n, n), dtype=bool)
causal_mask = np.tril(np.ones((n, n), dtype=bool))
_, full_weights = scaled_dot_product_attention(Q, K, V, mask=full_mask)
_, causal_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

mask_display = pd.DataFrame(causal_mask.astype(int), index=tokens, columns=tokens)
display(mask_display)

plot_attention(tokens, causal_weights, title="Decoder-style causal attention: future tokens are masked")


In [ ]:
# ============================================================
# 4.3 Static versus contextual representation of the word "charge"
# ============================================================

def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom > 0 else np.nan

billing_sentence = "the charge on my bill is wrong"
power_sentence = "the battery charge is low after charging"

semantic_vectors = defaultdict(lambda: rng.normal(scale=0.10, size=d_model))
semantic_vectors.update(base_vectors)
for tok in ["bill", "invoice", "refund", "billed", "subscription"]:
    semantic_vectors[tok] = rng.normal(scale=0.10, size=d_model) + billing_direction
for tok in ["battery", "charging", "charger", "power", "low"]:
    semantic_vectors[tok] = rng.normal(scale=0.10, size=d_model) + power_direction
semantic_vectors["charge"] = rng.normal(scale=0.10, size=d_model) + 0.5 * billing_direction + 0.5 * power_direction


def contextual_token_vector(sentence, target="charge", window=3):
    toks = word_tokenize(sentence)
    pos = toks.index(target)
    left = max(0, pos - window)
    right = min(len(toks), pos + window + 1)
    context = [t for i, t in enumerate(toks[left:right]) if i + left != pos]
    return semantic_vectors[target] + np.mean([semantic_vectors[t] for t in context], axis=0), context

static_similarity = cosine(semantic_vectors["charge"], semantic_vectors["charge"])
billing_vec, billing_context = contextual_token_vector(billing_sentence)
power_vec, power_context = contextual_token_vector(power_sentence)
contextual_similarity = cosine(billing_vec, power_vec)

comparison = pd.DataFrame([
    {"representation": "static token vector", "comparison": "charge vs. charge", "cosine_similarity": static_similarity, "context_used": "none"},
    {"representation": "contextual vector", "comparison": "billing charge vs. battery charge", "cosine_similarity": contextual_similarity, "context_used": f"billing: {billing_context}; power: {power_context}"},
])
display(comparison.round(3))


## 5. Pretraining, transfer, and model-family choices

A real foundation model is pretrained on massive corpora before it is adapted to a business task. This notebook cannot reproduce that scale, but it can show the operating logic. Encoder-style systems are usually strong for classification, extraction support, and embeddings. Decoder-style systems are usually strong for drafting, summarization, and structured generation. Business teams still supply the task definition, allowed schema, evaluation set, and governance procedure.


In [ ]:
# ============================================================
# 5.1 What pretraining gives and what business teams still supply
# ============================================================

pretraining_table = pd.DataFrame([
    {
        "pretraining_provides": "Broad competence with everyday language patterns",
        "not_provided_by_default": "Internal task definitions and decision rules",
        "organization_supplies": "Stable taxonomies, labeling guidelines, and edge-case policies",
    },
    {
        "pretraining_provides": "Context-sensitive representations",
        "not_provided_by_default": "Guaranteed correctness or groundedness",
        "organization_supplies": "Evaluation sets, evidence rules, and human review for risky cases",
    },
    {
        "pretraining_provides": "Reusable capability across many language tasks",
        "not_provided_by_default": "Fairness across segments, languages, and channels",
        "organization_supplies": "Stratified testing and disparity monitoring",
    },
    {
        "pretraining_provides": "Generative fluency for drafting and summarization",
        "not_provided_by_default": "Boundary adherence or source traceability",
        "organization_supplies": "Structured output schemas, citation requirements, and scope controls",
    },
])
display(pretraining_table)

pattern_library = pd.DataFrame([
    {"pattern": "Classification", "typical_output": "label", "best_used_for": "routing, dashboards, issue tracking", "evaluation_emphasis": "precision, recall, slice stability"},
    {"pattern": "Extraction", "typical_output": "structured fields", "best_used_for": "downstream analytics", "evaluation_emphasis": "schema validity and evidence spans"},
    {"pattern": "Similarity", "typical_output": "nearest neighbors", "best_used_for": "deduplication and theme grouping", "evaluation_emphasis": "coherence and drift"},
    {"pattern": "Summarization", "typical_output": "brief", "best_used_for": "reducing reading burden", "evaluation_emphasis": "faithfulness and omission risk"},
    {"pattern": "Guided assistance", "typical_output": "answer with evidence", "best_used_for": "bounded analyst support", "evaluation_emphasis": "grounding, boundaries, logging"},
])
display(pattern_library)


## 6. Encoder-style classification with a tiny Transformer

The next section trains a small Transformer encoder from scratch on synthetic issue labels. This is not how a production foundation model is built. It is a classroom-scale demonstration of how token IDs, embeddings, positional information, attention layers, pooling, and a classification head fit together. We first keep a TF-IDF baseline because a strong baseline is part of responsible evaluation.


In [ ]:
# ============================================================
# 6.1 Time-respecting split and sparse baseline
# ============================================================

cutoff = df["date"].quantile(0.75)
train_df = df[df["date"] < cutoff].copy()
test_df = df[df["date"] >= cutoff].copy()

print(f"Training rows: {len(train_df):,} | Test rows: {len(test_df):,} | Cutoff: {pd.Timestamp(cutoff).date()}")

tfidf_issue_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.95)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

tfidf_issue_model.fit(train_df["text_clean"], train_df["issue_type"])
tfidf_pred = tfidf_issue_model.predict(test_df["text_clean"])
print("TF-IDF baseline")
print(classification_report(test_df["issue_type"], tfidf_pred, zero_division=0))
plot_confusion(test_df["issue_type"], tfidf_pred, sorted(ISSUE_TYPES), "TF-IDF issue classification")


In [ ]:
# ============================================================
# 6.2 Build a small vocabulary and PyTorch dataset
# ============================================================

MAX_LEN = 42
MIN_FREQ = 2

counter = Counter()
for text in train_df["text_clean"]:
    counter.update(word_tokenize(text))

itos = ["<pad>", "<unk>"] + sorted([tok for tok, count in counter.items() if count >= MIN_FREQ])
stoi = {tok: i for i, tok in enumerate(itos)}
PAD_ID = stoi["<pad>"]
UNK_ID = stoi["<unk>"]

label_encoder = LabelEncoder()
label_encoder.fit(sorted(ISSUE_TYPES))


def encode_text(text, max_len=MAX_LEN):
    ids = [stoi.get(tok, UNK_ID) for tok in word_tokenize(text)[:max_len]]
    if len(ids) < max_len:
        ids = ids + [PAD_ID] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64)


class TextIssueDataset(Dataset):
    def __init__(self, frame):
        self.x = np.vstack([encode_text(t) for t in frame["text_clean"]])
        self.y = label_encoder.transform(frame["issue_type"])
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return torch.tensor(self.x[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)


train_dataset = TextIssueDataset(train_df)
test_dataset = TextIssueDataset(test_df)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Vocabulary size: {len(itos):,}")
print(f"Maximum sequence length: {MAX_LEN} tokens")
print(f"Labels: {list(label_encoder.classes_)}")


In [ ]:
# ============================================================
# 6.3 Define a tiny Transformer encoder classifier
# ============================================================

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, n_classes, d_model=48, n_heads=4, n_layers=1, max_len=MAX_LEN, dropout=0.10):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        z = self.token_embedding(x) + self.position_embedding(positions)
        padding_mask = x.eq(PAD_ID)
        encoded = self.encoder(z, src_key_padding_mask=padding_mask)
        valid = (~padding_mask).unsqueeze(-1).float()
        pooled = (encoded * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        pooled = self.norm(pooled)
        return self.classifier(pooled)


model = TinyTransformerClassifier(vocab_size=len(itos), n_classes=len(label_encoder.classes_)).to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

print(model)


In [ ]:
# ============================================================
# 6.4 Train and evaluate the tiny Transformer
# ============================================================

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    all_pred = []
    all_true = []
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        if is_train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        losses.append(loss.item() * len(y))
        all_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        all_true.extend(y.detach().cpu().numpy().tolist())
    avg_loss = sum(losses) / len(all_true)
    acc = accuracy_score(all_true, all_pred)
    f1 = f1_score(all_true, all_pred, average="macro", zero_division=0)
    return avg_loss, acc, f1

history = []
start = time.time()
for epoch in range(1, 7):
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, optimizer=optimizer)
    test_loss, test_acc, test_f1 = run_epoch(model, test_loader, optimizer=None)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_macro_f1": train_f1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "test_macro_f1": test_f1,
    })
    print(f"Epoch {epoch}: train f1={train_f1:.3f}, test f1={test_f1:.3f}")

print(f"Training time: {time.time() - start:.1f} seconds")
history_df = pd.DataFrame(history)
display(history_df.round(3))

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_macro_f1"], marker="o", label="train")
plt.plot(history_df["epoch"], history_df["test_macro_f1"], marker="o", label="test")
plt.title("Tiny Transformer issue classification learning curve")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 6.5 Inspect Transformer predictions and errors
# ============================================================

model.eval()
all_logits = []
with torch.no_grad():
    for x, _ in test_loader:
        all_logits.append(model(x.to(DEVICE)).cpu())
logits = torch.cat(all_logits, dim=0)
probs = torch.softmax(logits, dim=1).numpy()
transformer_pred_ids = probs.argmax(axis=1)
transformer_pred = label_encoder.inverse_transform(transformer_pred_ids)

print("Tiny Transformer")
print(classification_report(test_df["issue_type"], transformer_pred, zero_division=0))
plot_confusion(test_df["issue_type"], transformer_pred, list(label_encoder.classes_), "Tiny Transformer issue classification")

inspect = test_df.copy()
inspect["tfidf_pred"] = tfidf_pred
inspect["transformer_pred"] = transformer_pred
inspect["transformer_confidence"] = probs.max(axis=1)
inspect["transformer_correct"] = inspect["issue_type"].eq(inspect["transformer_pred"])

display(inspect.sort_values("transformer_confidence", ascending=True)[[
    "record_id", "issue_type", "tfidf_pred", "transformer_pred", "transformer_confidence", "text"
]].head(10))


## 7. Decoder-style structured outputs without calling a paid API

Decoder-style large language models are often used to generate summaries, answers, and structured fields. In production, the model call may happen through a vendor API or an internally hosted model. For this notebook, we will not call an external model. Instead, we create an API-ready prompt template and a rule-based stand-in that returns the same type of structured object. The important lesson is the schema: a business workflow should define allowed fields, valid labels, evidence spans, confidence, and human-review triggers.


In [ ]:
# ============================================================
# 7.1 Build an LLM-ready structured prompt template
# ============================================================

ALLOWED_LABEL_TEXT = ", ".join(sorted(ALLOWED_ISSUES))


def build_structured_prompt(customer_text):
    return f"""
You are a bounded text analytics assistant for customer feedback.
Return only a JSON object with these fields:
issue_type: one of [{ALLOWED_LABEL_TEXT}]
urgency: one of [low, medium, high]
product: product name or unknown
evidence_span: exact phrase from the customer text that supports the label
confidence: number from 0 to 1
needs_human_review: true or false

Rules:
- Use only the customer text below.
- If the text is outside customer feedback scope, return issue_type = out_of_scope.
- Do not invent policies, refunds, or facts.

Customer text:
{customer_text}
""".strip()

sample_text = test_df.sample(1, random_state=SEED)["text"].iloc[0]
print(build_structured_prompt(sample_text))


In [ ]:
# ============================================================
# 7.2 A local stand-in for structured output generation
# ============================================================

PRODUCT_PATTERN = re.compile(r"\b(" + "|".join(PRODUCT_NAMES) + r")\b", re.IGNORECASE)
KEYWORDS_BY_ISSUE = {
    "billing": ["charge", "bill", "billed", "invoice", "refund", "subscription fee", "renewal"],
    "power": ["battery", "charger", "charge", "power", "overheat", "drain", "hold a charge"],
    "delivery": ["delivery", "package", "shipping", "tracking", "delivered", "carrier"],
    "login": ["login", "log in", "password", "account", "two-factor", "verification", "locked"],
    "product_quality": ["cracked", "broken", "defect", "sensor", "scratch", "hinge", "quality", "damaged"],
    "praise": ["love", "great", "easy", "smooth", "recommend", "works as expected", "helpful"],
}


def issue_probabilities_from_tfidf(text):
    proba = tfidf_issue_model.predict_proba([normalize_text(text)])[0]
    classes = list(tfidf_issue_model.named_steps["clf"].classes_)
    return dict(zip(classes, proba))


def structured_output_stand_in(text, confidence_floor=0.38):
    probs = issue_probabilities_from_tfidf(text)
    best_issue = max(probs, key=probs.get)
    confidence = float(probs[best_issue])
    if confidence < confidence_floor:
        best_issue = "out_of_scope"
    product_match = PRODUCT_PATTERN.search(text)
    product = product_match.group(0) if product_match else "unknown"
    lowered = text.lower()
    urgency = "high" if any(w in lowered for w in ["urgent", "cancel", "manager", "safety", "today"]) else "medium"
    if best_issue == "praise":
        urgency = "low"
    evidence_keywords = KEYWORDS_BY_ISSUE.get(best_issue, [])
    evidence_span = first_matching_sentence(text, evidence_keywords) if evidence_keywords else first_matching_sentence(text, ["customer"])
    return {
        "issue_type": str(best_issue),
        "urgency": str(urgency),
        "product": str(product),
        "evidence_span": str(evidence_span),
        "confidence": round(float(confidence), 3),
        "needs_human_review": bool(urgency == "high" or confidence < 0.55 or best_issue == "out_of_scope"),
    }

examples_for_schema = [
    "The charge on my bill is wrong for NovaPhone. I need a refund today.",
    "My FitBand battery will not hold a charge and the charger is overheating.",
    "What is the best legal structure for my startup?",
]

structured_rows = []
for text in examples_for_schema:
    obj = structured_output_stand_in(text)
    problems = validate_structured_output(obj)
    structured_rows.append({"text": text, "output": obj, "schema_problems": problems})

display(pd.DataFrame(structured_rows))


## 8. Retrieval-supported question answering package

Retrieval-supported workflows reduce unsupported generation by retrieving approved evidence before producing an answer. In a production RAG system, a language model may compose the final response. Here, we use a transparent retrieval package that returns the query, the retrieved passages, the evidence, and a bounded answer. This keeps the grounding logic visible.


In [ ]:
# ============================================================
# 8.1 Build a simple retrieval index over approved passages
# ============================================================

retrieval_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), stop_words="english")
kb_matrix = retrieval_vectorizer.fit_transform(knowledge_base["title"] + " " + knowledge_base["passage"])


def retrieve_passages(query, top_k=3):
    q_vec = retrieval_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, kb_matrix).ravel()
    top_idx = np.argsort(sims)[::-1][:top_k]
    result = knowledge_base.iloc[top_idx].copy()
    result["similarity"] = sims[top_idx]
    return result[["doc_id", "topic", "title", "similarity", "passage"]]

query = "Customer says the battery charge drops fast and the charger gets hot. What should support collect?"
retrieved = retrieve_passages(query, top_k=3)
display(retrieved)


In [ ]:
# ============================================================
# 8.2 Create a grounded answer package from retrieved evidence
# ============================================================


def grounded_answer_package(query, top_k=3, min_similarity=0.08):
    retrieved = retrieve_passages(query, top_k=top_k)
    top = retrieved.iloc[0]
    if top["similarity"] < min_similarity:
        answer = "The approved knowledge base does not contain enough evidence to answer this question. Escalate for human review."
        needs_review = True
    else:
        answer = (
            f"Likely topic: {top['topic']}. Use {top['doc_id']} ({top['title']}) as the primary evidence. "
            f"Recommended next step: {top['passage']}"
        )
        needs_review = False
    package = {
        "query": query,
        "top_doc_ids": retrieved["doc_id"].tolist(),
        "top_similarity": round(float(top["similarity"]), 3),
        "answer": answer,
        "needs_human_review": needs_review,
    }
    return package, retrieved

queries = [
    "A customer reports a duplicate card charge and asks for a refund.",
    "The package is missing even though tracking says delivered.",
    "Can this assistant provide legal advice about employee termination?",
]

packages = []
for q in queries:
    package, _ = grounded_answer_package(q)
    packages.append(package)

display(pd.DataFrame(packages))


In [ ]:
# ============================================================
# 8.3 Grounding and source-traceability checks
# ============================================================


def grounding_checks(package, retrieved):
    answer = package["answer"]
    doc_ids_in_answer = [doc_id for doc_id in retrieved["doc_id"] if doc_id in answer]
    source_traceable = len(doc_ids_in_answer) > 0 or package["needs_human_review"]
    too_low_similarity = package["top_similarity"] < 0.08
    return {
        "source_traceable": source_traceable,
        "top_similarity": package["top_similarity"],
        "low_similarity_warning": too_low_similarity,
        "answer_length_words": len(answer.split()),
    }

check_rows = []
for q in queries:
    package, retrieved = grounded_answer_package(q)
    checks = grounding_checks(package, retrieved)
    check_rows.append({"query": q, **package, **checks})

display(pd.DataFrame(check_rows)[[
    "query", "top_doc_ids", "top_similarity", "source_traceable", "low_similarity_warning", "needs_human_review", "answer_length_words"
]])


## 9. Reliability tests for business failure modes

A fluent answer is not the same as a reliable answer. Business evaluation should include ordinary cases and targeted failure cases: negation, contrast, long input omission, ambiguous references, domain vocabulary, unsupported generation, and schema noncompliance. The small regression set below is designed to expose those risks.


In [ ]:
# ============================================================
# 9.1 Targeted failure-mode regression tests
# ============================================================

failure_tests = pd.DataFrame([
    {
        "case_id": "F01",
        "failure_mode": "negation and contrast",
        "text": "The delivery was not the problem. The charge on my bill is wrong and I need a refund.",
        "expected_issue": "billing",
    },
    {
        "case_id": "F02",
        "failure_mode": "ambiguous term",
        "text": "The battery charge is low, but there is no billing issue on my NovaPhone account.",
        "expected_issue": "power",
    },
    {
        "case_id": "F03",
        "failure_mode": "long input omission",
        "text": "Everything seemed fine. " * 25 + "At the end, the invoice charge is incorrect and the subscription fee doubled.",
        "expected_issue": "billing",
    },
    {
        "case_id": "F04",
        "failure_mode": "domain vocabulary",
        "text": "AtlasBook unit shows hinge wobble and sensor drift after firmware update AB-7712.",
        "expected_issue": "product_quality",
    },
    {
        "case_id": "F05",
        "failure_mode": "schema boundary",
        "text": "What are the tax implications of closing my business next year?",
        "expected_issue": "out_of_scope",
    },
    {
        "case_id": "F06",
        "failure_mode": "positive language with hidden issue",
        "text": "Great, another password reset loop. I love being locked out of my account again.",
        "expected_issue": "login",
    },
])

def predict_with_boundary(text):
    obj = structured_output_stand_in(text, confidence_floor=0.42)
    return obj

regression_rows = []
for _, row in failure_tests.iterrows():
    obj = predict_with_boundary(row["text"])
    regression_rows.append({
        "case_id": row["case_id"],
        "failure_mode": row["failure_mode"],
        "expected_issue": row["expected_issue"],
        "predicted_issue": obj["issue_type"],
        "confidence": obj["confidence"],
        "needs_human_review": obj["needs_human_review"],
        "pass": obj["issue_type"] == row["expected_issue"],
        "evidence_span": obj["evidence_span"],
    })

regression_results = pd.DataFrame(regression_rows)
display(regression_results)


In [ ]:
# ============================================================
# 9.2 Failure-mode summary and review queue
# ============================================================

failure_summary = regression_results.groupby("failure_mode").agg(
    n=("case_id", "count"),
    pass_rate=("pass", "mean"),
    review_rate=("needs_human_review", "mean"),
    mean_confidence=("confidence", "mean"),
).round(3).reset_index()
display(failure_summary)

review_queue = regression_results[(~regression_results["pass"]) | (regression_results["needs_human_review"])]
print(f"Regression cases requiring analyst review: {len(review_queue)}")
display(review_queue[["case_id", "failure_mode", "expected_issue", "predicted_issue", "confidence", "evidence_span"]])


In [ ]:
# ============================================================
# 9.3 Slice-level evaluation on the held-out business corpus
# ============================================================

test_eval = test_df.copy()
test_eval["predicted_issue"] = tfidf_pred
test_eval["correct"] = test_eval["issue_type"].eq(test_eval["predicted_issue"])

slice_rows = []
for group_col in ["channel", "segment", "risk_level"]:
    for value, g in test_eval.groupby(group_col):
        slice_rows.append({
            "slice_type": group_col,
            "slice_value": value,
            "n": len(g),
            "accuracy": g["correct"].mean(),
        })

slice_eval = pd.DataFrame(slice_rows).sort_values(["slice_type", "accuracy"])
display(slice_eval.round(3))

plt.figure(figsize=(8, 4))
plot_df = slice_eval[slice_eval["slice_type"] == "channel"]
plt.bar(plot_df["slice_value"], plot_df["accuracy"])
plt.title("Issue classification accuracy by channel")
plt.xlabel("Channel")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 10. Token budget, batch workflow design, and ROI simulation

Deployment is a workflow decision, not only a modeling decision. Batch workflows emphasize throughput, consistency, and versioning. Interactive workflows emphasize responsiveness, evidence display, and human review. Token count affects both. The following calculations use hypothetical rates so that the notebook remains vendor-neutral.


In [ ]:
# ============================================================
# 10.1 Hypothetical token budget and cost calculator
# ============================================================

pricing_policy = {
    "input_cost_per_1k_tokens": 0.002,   # hypothetical classroom value, not a vendor quote
    "output_cost_per_1k_tokens": 0.006,  # hypothetical classroom value, not a vendor quote
    "average_output_tokens": 90,
}

batch_plan = df.groupby("channel").agg(
    n_texts=("record_id", "count"),
    avg_input_tokens=("token_count", "mean"),
).reset_index()
batch_plan["expected_input_tokens"] = batch_plan["n_texts"] * batch_plan["avg_input_tokens"]
batch_plan["expected_output_tokens"] = batch_plan["n_texts"] * pricing_policy["average_output_tokens"]
batch_plan["hypothetical_input_cost"] = batch_plan["expected_input_tokens"] / 1000 * pricing_policy["input_cost_per_1k_tokens"]
batch_plan["hypothetical_output_cost"] = batch_plan["expected_output_tokens"] / 1000 * pricing_policy["output_cost_per_1k_tokens"]
batch_plan["hypothetical_total_cost"] = batch_plan["hypothetical_input_cost"] + batch_plan["hypothetical_output_cost"]
display(batch_plan.round(3))

plt.figure(figsize=(8, 4))
plt.bar(batch_plan["channel"], batch_plan["hypothetical_total_cost"])
plt.title("Hypothetical processing cost by channel")
plt.xlabel("Channel")
plt.ylabel("Cost in classroom units")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 10.2 Batch versus interactive workflow capacity planning
# ============================================================

workflow_options = pd.DataFrame([
    {"workflow": "nightly batch dashboard", "texts_per_day": 5000, "avg_input_tokens": 120, "avg_output_tokens": 25, "human_review_share": 0.05, "minutes_per_review": 2.5},
    {"workflow": "support triage assistant", "texts_per_day": 800, "avg_input_tokens": 180, "avg_output_tokens": 120, "human_review_share": 0.18, "minutes_per_review": 4.0},
    {"workflow": "analyst guided Q&A", "texts_per_day": 120, "avg_input_tokens": 900, "avg_output_tokens": 250, "human_review_share": 0.35, "minutes_per_review": 6.0},
])
workflow_options["daily_input_tokens"] = workflow_options["texts_per_day"] * workflow_options["avg_input_tokens"]
workflow_options["daily_output_tokens"] = workflow_options["texts_per_day"] * workflow_options["avg_output_tokens"]
workflow_options["daily_human_review_hours"] = (
    workflow_options["texts_per_day"] * workflow_options["human_review_share"] * workflow_options["minutes_per_review"] / 60
)
workflow_options["hypothetical_daily_cost"] = (
    workflow_options["daily_input_tokens"] / 1000 * pricing_policy["input_cost_per_1k_tokens"]
    + workflow_options["daily_output_tokens"] / 1000 * pricing_policy["output_cost_per_1k_tokens"]
)
display(workflow_options.round(2))


In [ ]:
# ============================================================
# 10.3 ROI simulation: value comes from changed workflow, not benchmark scores alone
# ============================================================

adoption_rates = np.linspace(0.10, 0.95, 18)
monthly_cases = 6000
manual_minutes_per_case = 3.5
automated_minutes_per_case = 0.8
qa_review_share = 0.15
qa_minutes_per_review = 4.0
hourly_labor_value = 35.0
monthly_platform_cost = 850.0

roi_rows = []
for adoption in adoption_rates:
    cases_using_system = monthly_cases * adoption
    gross_minutes_saved = cases_using_system * (manual_minutes_per_case - automated_minutes_per_case)
    qa_minutes = cases_using_system * qa_review_share * qa_minutes_per_review
    net_hours_saved = (gross_minutes_saved - qa_minutes) / 60
    net_value = net_hours_saved * hourly_labor_value - monthly_platform_cost
    roi_rows.append({
        "adoption_rate": adoption,
        "cases_using_system": cases_using_system,
        "net_hours_saved": net_hours_saved,
        "net_value_after_platform_cost": net_value,
    })

roi_df = pd.DataFrame(roi_rows)
display(roi_df.round(2).head())

plt.figure(figsize=(8, 4))
plt.plot(roi_df["adoption_rate"], roi_df["net_value_after_platform_cost"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Hypothetical ROI depends on adoption and workflow redesign")
plt.xlabel("Adoption rate")
plt.ylabel("Monthly net value in classroom units")
plt.tight_layout()
plt.show()


## 11. Governance checklist and system card

Prompts, retrieved passages, outputs, logs, model versions, and evaluation sets should be treated as governed data assets. A system card documents intended use, allowed inputs, output schema, evaluation, monitoring, and misuse boundaries. This is especially important when a language system is reused across teams.


In [ ]:
# ============================================================
# 11.1 Production-readiness checklist for transformer-based text analytics
# ============================================================

governance_checklist = pd.DataFrame([
    {"area": "Data scope", "decision": "Which sources are allowed and what must be redacted", "why_it_matters": "Prevents accidental processing of prohibited or high-risk content"},
    {"area": "Access control", "decision": "Who can submit inputs, view outputs, and change prompts or taxonomies", "why_it_matters": "Reduces leakage risk and supports accountability"},
    {"area": "Retention", "decision": "How long inputs, outputs, logs, and evaluations are stored", "why_it_matters": "Aligns with privacy, legal, and contractual obligations"},
    {"area": "Logging", "decision": "How outputs can be traced to inputs, evidence, and model version", "why_it_matters": "Enables investigation and regression testing"},
    {"area": "Versioning", "decision": "How prompts, models, retrieval indexes, and schemas are tracked", "why_it_matters": "Prevents silent behavior changes"},
    {"area": "Human review", "decision": "Which outputs require approval or escalation", "why_it_matters": "Creates guardrails where mistakes are expensive"},
    {"area": "Monitoring", "decision": "Which KPIs, slices, and drift signals are reviewed", "why_it_matters": "Keeps performance stable as language and products change"},
    {"area": "Third-party risk", "decision": "Where data is processed and which contractual controls apply", "why_it_matters": "Matches deployment to organizational risk tolerance"},
])
display(governance_checklist)


In [ ]:
# ============================================================
# 11.2 Create a compact Transformer/LLM system card
# ============================================================

system_card = pd.DataFrame([
    {"field": "Intended use", "entry": "Classify customer feedback, extract structured fields, retrieve approved policy passages, and prepare bounded triage summaries."},
    {"field": "Not intended for", "entry": "Legal, medical, financial, employment, or other high-stakes advice outside the approved knowledge base."},
    {"field": "Input unit", "entry": "One customer feedback text or one bounded analyst query."},
    {"field": "Allowed output schema", "entry": "issue_type, urgency, product, evidence_span, confidence, needs_human_review."},
    {"field": "Model patterns", "entry": "Sparse baseline, tiny educational Transformer classifier, local structured-output stand-in, and TF-IDF retrieval package."},
    {"field": "Evaluation set", "entry": "Time-respecting holdout plus targeted regression tests for negation, long inputs, ambiguity, domain vocabulary, and boundaries."},
    {"field": "Human review triggers", "entry": "High urgency, low confidence, out-of-scope queries, low retrieval similarity, or failed regression tests."},
    {"field": "Monitoring", "entry": "Track accuracy by channel, segment, risk level, token lengths, retrieval similarity, schema validity, and review volume."},
    {"field": "Known limitations", "entry": "Synthetic demonstration data, no real pretrained model weights, no multilingual benchmark, no real privacy or vendor-risk assessment."},
])

display(system_card)
system_card.to_csv(OUTPUT_DIR / "ch24_transformer_llm_system_card.csv", index=False)
print(f"Saved system card to {OUTPUT_DIR / 'ch24_transformer_llm_system_card.csv'}")


## Exercises

1. Increase `MAX_LEN` from 42 to 64. Does the tiny Transformer improve on the long-input cases, or does the synthetic corpus make little difference?
2. Modify `KEYWORDS_BY_ISSUE` so that the word `charge` alone does not trigger billing. How does this affect ambiguous billing versus power cases?
3. Add five new regression tests for sarcasm, mixed sentiment, and multiple issues in the same message.
4. Change the retrieval `min_similarity` threshold. Which queries become human-review cases?
5. Replace the TF-IDF retriever with dense SVD vectors using `TruncatedSVD`. Compare the top retrieved passages.
6. Add a schema field called `recommended_action` and update `validate_structured_output` so it checks allowed actions.
7. Create a weekly dashboard of high-risk language-system outputs by channel.
8. Write a short system-card paragraph explaining why a high benchmark score does not prove that an LLM workflow is ready for deployment.


In [ ]:
# ============================================================
# Optional exercise starter: dense SVD retrieval instead of sparse TF-IDF retrieval
# ============================================================

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

svd = TruncatedSVD(n_components=min(5, kb_matrix.shape[1] - 1), random_state=SEED)
kb_dense = normalize(svd.fit_transform(kb_matrix))


def retrieve_passages_dense(query, top_k=3):
    q_sparse = retrieval_vectorizer.transform([query])
    q_dense = normalize(svd.transform(q_sparse))
    sims = (q_dense @ kb_dense.T).ravel()
    top_idx = np.argsort(sims)[::-1][:top_k]
    result = knowledge_base.iloc[top_idx].copy()
    result["dense_similarity"] = sims[top_idx]
    return result[["doc_id", "topic", "title", "dense_similarity", "passage"]]

exercise_query = "The customer cannot access the account after verification and password reset."
print("Sparse retrieval")
display(retrieve_passages(exercise_query, top_k=3))
print("Dense SVD retrieval")
display(retrieve_passages_dense(exercise_query, top_k=3))
